# Dense SHD comparison: LIF, PIF, and heterogeneous PIF

This notebook is a clean, standalone reproduction of the final dense three-hidden-layer SHD experiment. It uses the production classes in `stork.periodic_reset`, does **not** prune weights or neurons, and runs each model once with seed 42.

It answers three questions:

1. Does the new periodic-reset initialization preserve the fluctuation regime in a simple Poisson-driven network with spiking disabled?
2. Do homogeneous and heterogeneous PIF models reach performance comparable to the LIF baseline?
3. Do the dense PIF models use fewer effective operations per test sample than LIF?

The final experiment uses exactly 349 time steps. The historical readouts are retained: LIF uses the standard leaky `ReadoutGroup`, while both PIF models use `NonLeakyReadoutGroup`. EFLOPs include the readout and are reported as connection, neuron/readout, and total operations per test sample. The local default is 50 epochs; set `PIF_EPOCHS=500` to restore the historical training length.

## 1. Environment

The next cell locates the repository and installs only missing runtime dependencies. `randman` is installed from its official repository because Stork imports it but it is not published on PyPI.

The default device policy is backend-agnostic: CUDA is selected first when available, then Apple MPS, then CPU. Set `PIF_DEVICE=mps`, `PIF_DEVICE=cpu`, or for example `PIF_DEVICE=cuda:0` to override it. Training defaults to 50 epochs and can be overridden with `PIF_EPOCHS`. Set `PIF_NOTEBOOK_SMOKE=1` before starting the kernel for a structural smoke run.

In [ ]:
from pathlib import Path
import importlib.util
import os
import random
import subprocess
import sys


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "stork" / "periodic_reset").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find the repository root. Start Jupyter inside this repository."
    )


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

missing = []
if importlib.util.find_spec("h5py") is None:
    missing.append("h5py")
if importlib.util.find_spec("pandas") is None:
    missing.append("pandas")
if importlib.util.find_spec("randman") is None:
    missing.append("git+https://github.com/fzenke/randman.git")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

print(f"Repository: {REPO_ROOT}")

In [ ]:
import gzip
import hashlib
import shutil
import urllib.request
from collections import OrderedDict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import stork
from stork.connections import Connection
from stork.datasets import DatasetView, HDF5Dataset
from stork.generators import StandardGenerator
from stork.initializers import (
    DistInitializer,
    FluctuationDrivenCenteredNormalInitializer,
)
from stork.loss_stacks import MaxOverTimeCrossEntropy
from stork.models import RecurrentSpikingModel
from stork.nodes import InputGroup, LIFGroup, ReadoutGroup
from stork.optimizers import SMORMS3
from stork.periodic_reset import (
    EffectiveFlopsCounter,
    HeterogeneousPIFGroup,
    NonLeakyReadoutGroup,
    PIFGroup,
    PeriodicResetFluctuationDrivenInitializer,
)


SEED = 42
DT = 2e-3
N_STEPS = 349                  # explicit: do not recompute from floating-point division
DURATION = 0.7
N_INPUTS = 700
N_CLASSES = 20
N_HIDDEN_LAYERS = 3
N_HIDDEN_UNITS = 128
BATCH_SIZE = 400
REFERENCE_EPOCHS = 500
EPOCHS = int(os.environ.get("PIF_EPOCHS", "50"))
LEARNING_RATE = 5e-3
NU = 15.8
TARGET_MEAN = 0.0
TARGET_STD = 1.0

LIF_TAU_MEM = 20e-3
LIF_TAU_SYN = 10e-3
PIF_TAU = 40e-3
HETERO_CONCENTRATION = 2.0
READOUT_TAU_MEM = 0.7
READOUT_INITIAL_STATE = -1e-3

SMOKE_TEST = os.environ.get("PIF_NOTEBOOK_SMOKE", "0") == "1"
RUN_EPOCHS = 1 if SMOKE_TEST else EPOCHS
RUN_HIDDEN_LAYERS = 1 if SMOKE_TEST else N_HIDDEN_LAYERS
RUN_HIDDEN_UNITS = 16 if SMOKE_TEST else N_HIDDEN_UNITS
RUN_BATCH_SIZE = 8 if SMOKE_TEST else BATCH_SIZE

def select_device(requested=None):
    '''Resolve an explicit backend or choose the best available accelerator.'''
    requested = (requested or os.environ.get("PIF_DEVICE", "auto")).lower()
    if requested != "auto":
        device = torch.device(requested)
        if device.type == "cuda" and not torch.cuda.is_available():
            raise RuntimeError(f"Requested {requested}, but CUDA is unavailable")
        if device.type == "mps" and not torch.backends.mps.is_available():
            raise RuntimeError("Requested MPS, but torch.backends.mps.is_available() is false")
        return device
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = select_device()
# HDF5-backed datasets and spawned workers are a fragile combination on macOS.
# CUDA runs retain the historical four workers; MPS and CPU stay portable.
N_WORKERS = 4 if DEVICE.type == "cuda" and not SMOKE_TEST else 0
MODEL_LABELS = OrderedDict(
    lif="LIF",
    pif="PIF",
    hetero_pif="heterogeneous PIF",
)
MODEL_COLORS = {
    "lif": "tab:blue",
    "pif": "tab:orange",
    "hetero_pif": "tab:green",
}


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed()
print(f"Device: {DEVICE}")
print(f"Smoke mode: {SMOKE_TEST}")
print(f"Training epochs: {RUN_EPOCHS}")
if DEVICE.type == "cpu" and not SMOKE_TEST:
    print(f"Warning: CPU is supported, but 3 x {RUN_EPOCHS} epochs will be slow.")

## 2. SHD download and deterministic split

The files come from the [official Zenke Lab SHD distribution](https://zenkelab.org/resources/spiking-heidelberg-datasets-shd/). Downloads are streamed, checked against the official compressed-file MD5 sums, and decompressed into `data/datasets/hdspikes`. Existing decompressed files are reused.

The historical experiment used a random 90/10 training/validation split with no held-out speaker. The split below is performed once with seed 42 and shared by all three models.

In [ ]:
DATA_DIR = REPO_ROOT / "data" / "datasets" / "hdspikes"
SHD_FILES = {
    "shd_train.h5": {
        "url": "https://zenkelab.org/datasets/shd_train.h5.gz",
        "md5": "d47c9825dee33347913e8ce0f2be08b0",
    },
    "shd_test.h5": {
        "url": "https://zenkelab.org/datasets/shd_test.h5.gz",
        "md5": "3062a80ec0c5719404d5b02e166543b1",
    },
}


def md5sum(path, chunk_size=2**20):
    digest = hashlib.md5()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_shd_file(filename, spec):
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    destination = DATA_DIR / filename
    if destination.exists():
        print(f"Using existing {destination}")
        return destination

    archive = DATA_DIR / f"{filename}.gz"
    partial = DATA_DIR / f"{filename}.gz.part"
    if not archive.exists() or md5sum(archive) != spec["md5"]:
        partial.unlink(missing_ok=True)
        print(f"Downloading {spec['url']}")
        with urllib.request.urlopen(spec["url"]) as response, partial.open("wb") as out:
            shutil.copyfileobj(response, out, length=2**20)
        observed = md5sum(partial)
        if observed != spec["md5"]:
            partial.unlink(missing_ok=True)
            raise RuntimeError(
                f"MD5 mismatch for {filename}: expected {spec['md5']}, got {observed}"
            )
        partial.replace(archive)

    temporary = DATA_DIR / f"{filename}.tmp"
    print(f"Decompressing {archive.name}")
    with gzip.open(archive, "rb") as source, temporary.open("wb") as target:
        shutil.copyfileobj(source, target, length=2**20)
    temporary.replace(destination)
    return destination


dataset_paths = {
    name: download_shd_file(name, spec) for name, spec in SHD_FILES.items()
}
dataset_paths

In [ ]:
dataset_kwargs = dict(
    nb_steps=N_STEPS,
    nb_units=N_INPUTS,
    time_scale=1.0 / DT,
    unit_scale=1.0,
    preload=True,
    precompute_dense=False,
    unit_permutation=None,
)

full_train_dataset = HDF5Dataset(dataset_paths["shd_train.h5"], **dataset_kwargs)
test_dataset = HDF5Dataset(dataset_paths["shd_test.h5"], **dataset_kwargs)

split_rng = np.random.RandomState(SEED)
indices = np.arange(len(full_train_dataset))
split_rng.shuffle(indices)
split_at = int(0.9 * len(indices))
train_dataset = DatasetView(full_train_dataset, indices[:split_at])
valid_dataset = DatasetView(full_train_dataset, indices[split_at:])

if SMOKE_TEST:
    train_dataset = torch.utils.data.Subset(train_dataset, range(min(32, len(train_dataset))))
    valid_dataset = torch.utils.data.Subset(valid_dataset, range(min(16, len(valid_dataset))))
    test_dataset = torch.utils.data.Subset(test_dataset, range(min(16, len(test_dataset))))

print(
    f"train={len(train_dataset)}, validation={len(valid_dataset)}, "
    f"test={len(test_dataset)}, steps={N_STEPS}"
)

## 3. Exact dense model definitions

All models have three feed-forward hidden layers of 128 neurons, no recurrent connections, no regularization, and no pruning. They use SMORMS3 at learning rate 0.005 and max-over-time cross-entropy.

- LIF: `tau_mem=20 ms`, `tau_syn=10 ms`, the standard fluctuation-driven initializer, and a leaky readout.
- PIF: homogeneous `tau=40 ms`, the new periodic-reset initializer, and a non-leaky readout.
- heterogeneous PIF: Gamma-distributed periods with mean `40 ms` and concentration `2`, the same new initializer parameterized by the population mean period, and a non-leaky readout.

In [ ]:
def make_hidden_group(kind, shape, store_membrane=False, disable_spiking=False):
    store_sequences = ["mem"] if store_membrane else None
    if kind == "lif":
        activation = NeverSpike if disable_spiking else stork.activations.SuperSpike
        return LIFGroup(
            shape=shape,
            tau_mem=LIF_TAU_MEM,
            tau_syn=LIF_TAU_SYN,
            activation=activation,
            store_sequences=store_sequences,
        )

    threshold = 1e6 if disable_spiking else 1.0
    common = dict(
        shape=shape,
        tau=PIF_TAU,
        threshold=threshold,
        activation=stork.activations.SuperSpike,
        store_sequences=store_sequences,
    )
    if kind == "pif":
        return PIFGroup(**common)
    if kind == "hetero_pif":
        return HeterogeneousPIFGroup(
            concentration=HETERO_CONCENTRATION,
            **common,
        )
    raise ValueError(f"Unknown model kind: {kind}")


def make_hidden_initializer(kind):
    if kind == "lif":
        return FluctuationDrivenCenteredNormalInitializer(
            sigma_u=TARGET_STD,
            nu=NU,
            timestep=DT,
            alpha=0.9,
        )
    return PeriodicResetFluctuationDrivenInitializer(
        mu_u=TARGET_MEAN,
        sigma_u=TARGET_STD,
        nu=NU,
        tau=PIF_TAU,
        center_weights=True,
    )


def make_generator():
    return StandardGenerator(
        nb_workers=N_WORKERS,
        persistent_workers=N_WORKERS > 0,
    )


def build_training_model(kind):
    set_seed(SEED)
    model = RecurrentSpikingModel(
        batch_size=RUN_BATCH_SIZE,
        nb_time_steps=N_STEPS,
        nb_inputs=N_INPUTS,
        device=DEVICE,
        dtype=torch.float32,
    )
    input_group = model.add_group(InputGroup(N_INPUTS))
    upstream = input_group
    hidden_groups = []
    initializer = make_hidden_initializer(kind)

    for _ in range(RUN_HIDDEN_LAYERS):
        hidden = model.add_group(make_hidden_group(kind, RUN_HIDDEN_UNITS))
        connection = model.add_connection(Connection(upstream, hidden))
        connection.init_parameters(initializer)
        hidden_groups.append(hidden)
        upstream = hidden

    if kind == "lif":
        readout = model.add_group(
            ReadoutGroup(
                shape=N_CLASSES,
                tau_mem=READOUT_TAU_MEM,
                tau_syn=LIF_TAU_SYN,
                initial_state=READOUT_INITIAL_STATE,
            )
        )
    else:
        readout = model.add_group(
            NonLeakyReadoutGroup(
                shape=N_CLASSES,
                tau_mem=READOUT_TAU_MEM,
                initial_state=READOUT_INITIAL_STATE,
            )
        )

    readout_connection = model.add_connection(
        Connection(upstream, readout, flatten_input=True)
    )
    DistInitializer(
        dist=torch.distributions.Normal(0.0, 1.0),
        scaling="1/sqrt(k)",
    ).initialize(readout_connection)

    model.configure(
        input=input_group,
        output=readout,
        loss_stack=MaxOverTimeCrossEntropy(),
        optimizer=SMORMS3,
        optimizer_kwargs=dict(lr=LEARNING_RATE),
        generator=make_generator(),
        time_step=DT,
        wandb=None,
    )
    model.experiment_kind = kind
    model.experiment_hidden_groups = hidden_groups
    model.experiment_readout = readout
    return model


config_table = pd.DataFrame(
    [
        {
            "model": MODEL_LABELS[kind],
            "hidden dynamics": (
                "LIF (20/10 ms)" if kind == "lif" else
                "PIF (40 ms)" if kind == "pif" else
                "Gamma PIF (mean 40 ms, concentration 2)"
            ),
            "readout": "leaky" if kind == "lif" else "non-leaky",
            "hidden initializer": "LIF FDI" if kind == "lif" else "periodic-reset FDI",
            "hidden layers": RUN_HIDDEN_LAYERS,
            "units/layer": RUN_HIDDEN_UNITS,
            "epochs": RUN_EPOCHS,
            "pruning": False,
        }
        for kind in MODEL_LABELS
    ]
)
config_table

## 4. Separate initialization experiment

A one-layer network receives independent Poisson input at 15.8 Hz through 700 afferents. It uses the same hidden-neuron parameters and initializers as the final classifiers. Spiking is disabled (`NeverSpike` for LIF and an effectively infinite PIF threshold), so resets caused by output spikes cannot contaminate the membrane statistics.

For each model, the temporal mean and standard deviation are calculated separately for every neuron after burn-in, across batches and repetitions. The reported empirical values are the averages of those per-neuron statistics, matching the fluctuation-regime calculation used in the earlier experiments.

The theoretical values use the same discrete simulation as the experiment. Each input bin is Bernoulli, `rand < p`, with $p=\nu\Delta t=0.0316$: its mean is $p$, but its variance is $p(1-p)$ rather than the ideal Poisson-count variance $p$. This produces the small finite-bin correction $1-p$. For PIF with an integer reset period $P$, the mean integration age is $(P-1)/2$ because the reset step itself has age zero. Heterogeneous PIF calculates a theoretical standard deviation for every sampled $P_i$ and averages those values, exactly matching the empirical `mean(per-neuron std)` statistic. The requested continuous-time target remains mean 0 and standard deviation 1, while the discrete theory explains the small finite-timestep offset from that target.

In [ ]:
class NeverSpike:
    @staticmethod
    def apply(x):
        return torch.zeros_like(x)


INIT_DURATION = 2.0
INIT_STEPS = int(INIT_DURATION / DT)
INIT_BURN_IN_STEPS = int(0.2 / DT)
INIT_BATCH_SIZE = 4 if SMOKE_TEST else 16
INIT_REPETITIONS = 1 if SMOKE_TEST else 8


def build_initialization_model(kind):
    set_seed(SEED)
    model = RecurrentSpikingModel(
        batch_size=INIT_BATCH_SIZE,
        nb_time_steps=INIT_STEPS,
        nb_inputs=N_INPUTS,
        device=DEVICE,
        dtype=torch.float32,
    )
    input_group = model.add_group(InputGroup(N_INPUTS))
    hidden = model.add_group(
        make_hidden_group(
            kind,
            N_HIDDEN_UNITS,
            store_membrane=True,
            disable_spiking=True,
        )
    )
    connection = model.add_connection(Connection(input_group, hidden))
    initializer = make_hidden_initializer(kind)
    connection.init_parameters(initializer)
    model.configure(
        input=input_group,
        output=hidden,
        loss_stack=MaxOverTimeCrossEntropy(),
        optimizer=SMORMS3,
        optimizer_kwargs=dict(lr=LEARNING_RATE),
        generator=StandardGenerator(nb_workers=0, persistent_workers=False),
        time_step=DT,
    )
    return model, hidden, connection, initializer


def calculate_theoretical_statistics(kind, hidden, connection, initializer):
    fan_in = connection.op.weight.shape[1]
    mu_weight, sigma_weight = initializer._get_weight_parameters_con(connection)
    weight_second_moment = sigma_weight**2 + mu_weight**2
    spike_probability = NU * DT
    if kind == "lif":
        epsilon_bar, epsilon_hat = initializer._calc_epsilon(hidden)
        mean = fan_in * NU * epsilon_bar * mu_weight
        variance = (
            fan_in
            * NU
            * (1.0 - spike_probability)
            * epsilon_hat
            * weight_second_moment
        )
        standard_deviation = variance**0.5
    else:
        # Move first, then promote: MPS cannot perform a float64 conversion on-device.
        periods = hidden.period_steps.detach().cpu().to(torch.float64)
        mean_age = torch.clamp(periods - 1.0, min=0.0) / 2.0
        per_neuron_mean = fan_in * spike_probability * mu_weight * mean_age
        per_neuron_variance = (
            fan_in
            * spike_probability
            * (1.0 - spike_probability)
            * weight_second_moment
            * mean_age
        )
        mean = per_neuron_mean.mean().item()
        standard_deviation = per_neuron_variance.sqrt().mean().item()
    return float(mean), float(standard_deviation)


def validate_initialization(kind):
    model, hidden, connection, initializer = build_initialization_model(kind)
    theoretical_mean, theoretical_std = calculate_theoretical_statistics(
        kind, hidden, connection, initializer
    )
    probability = NU * DT
    samples_per_neuron = 0
    per_neuron_sum = torch.zeros(N_HIDDEN_UNITS, dtype=torch.float64)
    per_neuron_square_sum = torch.zeros(N_HIDDEN_UNITS, dtype=torch.float64)
    observed_spikes = 0

    set_seed(SEED)
    model.eval()
    with torch.no_grad():
        for _ in range(INIT_REPETITIONS):
            poisson_input = (
                torch.rand(INIT_BATCH_SIZE, INIT_STEPS, N_INPUTS) < probability
            ).to(torch.float32)
            model.forward_pass(poisson_input, cur_batch_size=INIT_BATCH_SIZE)
            membrane = hidden.get_state_sequence("mem")[:, INIT_BURN_IN_STEPS:]
            observed_spikes += int(hidden.get_out_sequence().ne(0).sum().item())
            samples_per_neuron += membrane.shape[0] * membrane.shape[1]
            # Accumulate on the host in float64. MPS does not implement float64,
            # so the device-side reductions intentionally use the model dtype.
            per_neuron_sum += membrane.sum(dim=(0, 1)).cpu().to(torch.float64)
            per_neuron_square_sum += membrane.square().sum(
                dim=(0, 1)
            ).cpu().to(torch.float64)
            del poisson_input, membrane

    per_neuron_mean = per_neuron_sum / samples_per_neuron
    per_neuron_variance = torch.clamp(
        per_neuron_square_sum / samples_per_neuron - per_neuron_mean.square(),
        min=0.0,
    )
    empirical_mean = float(per_neuron_mean.mean().item())
    empirical_std = float(per_neuron_variance.sqrt().mean().item())
    sampled_tau_ms = (
        float(hidden.tau.float().mean().item() * 1e3)
        if hasattr(hidden, "tau")
        else np.nan
    )
    return {
        "kind": kind,
        "model": MODEL_LABELS[kind],
        "theoretical mean": theoretical_mean,
        "empirical mean": empirical_mean,
        "absolute mean error": abs(empirical_mean - theoretical_mean),
        "theoretical std": theoretical_std,
        "empirical std": empirical_std,
        "relative std error (%)": 100 * abs(empirical_std - theoretical_std) / theoretical_std,
        "sampled mean tau (ms)": sampled_tau_ms,
        "observed spikes": observed_spikes,
    }


initialization_results = pd.DataFrame(
    [validate_initialization(kind) for kind in MODEL_LABELS]
)
initialization_results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
x = np.arange(len(initialization_results))
labels = initialization_results["model"]

axes[0].plot(
    x,
    initialization_results["theoretical mean"],
    color="0.25",
    linestyle="--",
    marker="x",
    label="discrete theory",
)
axes[0].scatter(
    x,
    initialization_results["empirical mean"],
    s=65,
    color=[MODEL_COLORS[k] for k in initialization_results["kind"]],
    label="empirical",
)
for i, value in enumerate(initialization_results["empirical mean"]):
    axes[0].annotate(f"{value:.3f}", (i, value), xytext=(0, 8), textcoords="offset points", ha="center")
axes[0].set_title("Membrane mean")
axes[0].set_ylabel("membrane potential")
axes[0].set_xticks(x, labels, rotation=15, ha="right")
axes[0].legend()

axes[1].plot(
    x,
    initialization_results["theoretical std"],
    color="0.25",
    linestyle="--",
    marker="x",
    label="discrete theory",
)
axes[1].scatter(
    x,
    initialization_results["empirical std"],
    s=65,
    color=[MODEL_COLORS[k] for k in initialization_results["kind"]],
    label="empirical",
)
for i, value in enumerate(initialization_results["empirical std"]):
    axes[1].annotate(f"{value:.3f}", (i, value), xytext=(0, 8), textcoords="offset points", ha="center")
axes[1].set_title("Membrane standard deviation")
axes[1].set_ylabel("membrane potential")
axes[1].set_xticks(x, labels, rotation=15, ha="right")
axes[1].legend()
plt.show()

## 5. Train each dense classifier once

The three models share the dataset split and seed. The seed is reset before model construction and immediately before training so that the run is deterministic and minibatch shuffling starts from the same RNG state. No model selection, pruning, or repeated run is performed. Test loss and accuracy are accumulated with batch-size weighting, so the smaller final batch contributes exactly its number of samples.

The local run uses 50 epochs for each model, which is enough for this behavioral comparison and takes roughly 1 hour 40 minutes on the tested M2. Set `PIF_EPOCHS=500` before starting the kernel only when the historical training length is required.

For each model, the notebook checks `data/periodic_reset/weights`. If its weight file exists, it is loaded and training is skipped. Otherwise the model is trained once and its weights are saved. Saving uses `torch.save` and overwrites the corresponding file. Only model weights are stored—no history, optimizer, metrics, or checkpoint state. The repository already ignores `data/`, so generated weights are never part of the contribution.

In [ ]:
WEIGHTS_DIR = REPO_ROOT / "data" / "periodic_reset" / "weights"


def evaluate_sample_weighted(model, dataset):
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    model.eval()
    model.prepare_data(dataset)
    with torch.no_grad():
        for local_x, local_y in model.data_generator(dataset, shuffle=False):
            batch_size = len(local_y)
            output = model.forward_pass(local_x, cur_batch_size=batch_size)
            model.get_total_loss(output, local_y, regularized=False)
            predictions = model.loss_stack.predict(output)
            targets = local_y.to(model.device)
            total_loss += float(model.out_loss.item()) * batch_size
            total_correct += int(predictions.eq(targets).sum().item())
            total_samples += batch_size
    return {
        "loss": total_loss / total_samples,
        "accuracy": total_correct / total_samples,
        "samples": total_samples,
    }


def model_weights_path(kind):
    prefix = "smoke_" if SMOKE_TEST else ""
    return WEIGHTS_DIR / f"{prefix}{kind}.pth"


def save_model_weights(kind, model):
    WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
    path = model_weights_path(kind)
    state_dict = {
        name: value.detach().cpu()
        for name, value in model.state_dict().items()
    }
    torch.save(state_dict, path)  # Standard overwrite behavior.
    print(f"Saved weights: {path}")


def load_model_weights(kind, model):
    path = model_weights_path(kind)
    if not path.exists():
        return False

    try:
        state_dict = torch.load(path, map_location=DEVICE, weights_only=True)
    except TypeError:  # Compatibility with older PyTorch releases.
        state_dict = torch.load(path, map_location=DEVICE)

    model.load_state_dict(state_dict, strict=True)
    # period_steps and offset are derived, non-persistent buffers. Recompute
    # them after loading tau and phase from PIF weights.
    for hidden in model.experiment_hidden_groups:
        if isinstance(hidden, HeterogeneousPIFGroup):
            hidden.configure(
                model.batch_size,
                model.nb_time_steps,
                model.time_step,
                model.device,
                model.dtype,
            )
    print(f"Loaded weights: {path}")
    return True

In [ ]:
models = {}
histories = {}
test_rows = []

for kind, label in MODEL_LABELS.items():
    print(f"\n--- {label} ---")
    model = build_training_model(kind)
    loaded = load_model_weights(kind, model)
    if not loaded:
        print(f"Training {label} for {RUN_EPOCHS} epochs")
        set_seed(SEED)
        history = model.fit_validate(
            train_dataset,
            valid_dataset,
            nb_epochs=RUN_EPOCHS,
            verbose=True,
        )
        histories[kind] = history
        save_model_weights(kind, model)
    else:
        print(f"Skipping training for {label}")

    test_metrics = evaluate_sample_weighted(model, test_dataset)
    models[kind] = model
    test_rows.append(
        {
            "kind": kind,
            "model": label,
            "test loss": test_metrics["loss"],
            "test accuracy": test_metrics["accuracy"],
            "test samples": test_metrics["samples"],
        }
    )
    print(f"{label} test accuracy: {test_metrics['accuracy']:.4f}")

test_results = pd.DataFrame(test_rows)
test_results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for kind, history in histories.items():
    label = MODEL_LABELS[kind]
    epochs = np.arange(1, len(history["loss"]) + 1)
    axes[0].plot(epochs, history["loss"], color=MODEL_COLORS[kind], label=f"{label} train")
    axes[0].plot(epochs, history["val_loss"], color=MODEL_COLORS[kind], linestyle="--", label=f"{label} validation")
    axes[1].plot(epochs, history["acc"], color=MODEL_COLORS[kind], label=f"{label} train")
    axes[1].plot(epochs, history["val_acc"], color=MODEL_COLORS[kind], linestyle="--", label=f"{label} validation")

axes[0].set(title="Cross-entropy", xlabel="epoch", ylabel="loss")
axes[1].set(title="Classification accuracy", xlabel="epoch", ylabel="accuracy")
axes[1].set_ylim(0, 1)
for axis in axes:
    axis.grid(alpha=0.25)
if histories:
    axes[1].legend(ncol=2, fontsize=8)
else:
    axes[0].text(0.5, 0.5, "Models loaded: histories are not saved", ha="center")
    axes[1].text(0.5, 0.5, "Models loaded: histories are not saved", ha="center")
plt.show()

## 6. Effective operations on the full test set

EFLOPs are counted from the actual test traces and final dense weights. The counter includes all four feed-forward connections, all three hidden layers, and the readout. Counts are accumulated batch by batch and divided by the exact number of test samples.

The updating masks follow the historical experiment convention: a postsynaptic neuron is marked as updating at a time step when at least one active presynaptic spike has a nonzero connection to it. Since this experiment is dense and unpruned, the common dense case is handled without a large auxiliary matrix multiplication.

Briefly, the counter adds two components:

- **Connections:** one effective operation for every nonzero outgoing weight activated by a presynaptic spike.
- **Neurons/readout:** operations required only while a state is nonzero or receives input. The historical convention assigns five base operations to an active LIF state and two to an active PIF or non-leaky accumulator; PIF also pays for active scheduled resets. Direct assignment into a previously inactive state is counted as a saving.

Thus, `total EFLOPs = connection EFLOPs + neuron/readout EFLOPs`. These are analytical event-driven operation counts, not measured MPS runtime or energy: the current PyTorch implementation still executes dense tensor operations. With no pruning, PIF savings come mainly from cheaper neuron dynamics and any difference in observed spike activity.

In [ ]:
def postsynaptic_updating_mask(spikes, weights, epsilon=0.0):
    active_weights = weights.detach().abs() > epsilon
    active_spikes = spikes.detach().ne(0)
    if bool(active_weights.all().item()):
        any_spike = active_spikes.any(dim=-1, keepdim=True)
        return any_spike.expand(*active_spikes.shape[:-1], active_weights.shape[0])
    return torch.matmul(
        active_spikes.to(torch.float32),
        active_weights.T.to(device=spikes.device, dtype=torch.float32),
    ).gt(0)


def count_test_eflops(model, dataset):
    counter = EffectiveFlopsCounter()
    for hidden in model.experiment_hidden_groups:
        if "mem" not in hidden.store_state_sequences:
            hidden.store_state_sequences.append("mem")

    connection_total = 0
    neuron_total = 0
    sample_total = 0
    model.eval()
    model.prepare_data(dataset)

    with torch.no_grad():
        for batch_index, (local_x, local_y) in enumerate(
            model.data_generator(dataset, shuffle=False), start=1
        ):
            model.forward_pass(local_x, cur_batch_size=len(local_y))
            source_groups = [model.input_group, *model.experiment_hidden_groups]
            source_spikes = [group.get_out_sequence() for group in source_groups]

            connection_pairs = []
            updating_masks = []
            for spikes, connection in zip(source_spikes, model.connections):
                weights = connection.op.weight
                connection_pairs.append((spikes, weights))
                updating_masks.append(postsynaptic_updating_mask(spikes, weights))

            pif_layers = []
            lif_layers = []
            for hidden, updating in zip(
                model.experiment_hidden_groups, updating_masks[:-1]
            ):
                membrane = hidden.get_state_sequence("mem")
                if isinstance(hidden, HeterogeneousPIFGroup):
                    reset_mask = counter.periodic_reset_mask(
                        hidden.period_steps,
                        hidden.offset,
                        num_steps=N_STEPS,
                        batch_size=len(local_y),
                    )
                    pif_layers.append((membrane, updating, reset_mask))
                else:
                    lif_layers.append((membrane, updating))

            readout_output = model.experiment_readout.get_out_sequence()
            non_leaky_readouts = []
            if isinstance(model.experiment_readout, NonLeakyReadoutGroup):
                non_leaky_readouts.append((readout_output, updating_masks[-1]))
            else:
                lif_layers.append((readout_output, updating_masks[-1]))

            batch_count = counter.count(
                connections=connection_pairs,
                pif_layers=pif_layers,
                lif_layers=lif_layers,
                non_leaky_readouts=non_leaky_readouts,
            )
            connection_total += batch_count.connection_operations
            neuron_total += batch_count.neuron_operations
            sample_total += len(local_y)
            print(
                f"batch {batch_index}: {sample_total}/{len(dataset)} samples",
                end="\r",
            )

    print()
    return {
        "connection EFLOPs/sample": connection_total / sample_total,
        "neuron + readout EFLOPs/sample": neuron_total / sample_total,
        "total EFLOPs/sample": (connection_total + neuron_total) / sample_total,
        "samples counted": sample_total,
    }


eflop_rows = []
for kind, label in MODEL_LABELS.items():
    print(f"Counting {label}")
    row = {"kind": kind, "model": label}
    row.update(count_test_eflops(models[kind], test_dataset))
    eflop_rows.append(row)

eflop_results = pd.DataFrame(eflop_rows)
final_results = test_results.merge(eflop_results, on=["kind", "model"])
final_results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
x = np.arange(len(final_results))
colors = [MODEL_COLORS[k] for k in final_results["kind"]]

accuracy_bars = axes[0].bar(x, final_results["test accuracy"], color=colors)
axes[0].set(title="Final SHD test accuracy", ylabel="accuracy")
axes[0].set_ylim(0, 1)
axes[0].set_xticks(x, final_results["model"], rotation=15, ha="right")
axes[0].bar_label(accuracy_bars, fmt="%.3f", padding=3)

connection_values = final_results["connection EFLOPs/sample"]
neuron_values = final_results["neuron + readout EFLOPs/sample"]
axes[1].bar(x, connection_values, label="connections", color="0.55")
total_bars = axes[1].bar(
    x,
    neuron_values,
    bottom=connection_values,
    label="neurons + readout",
    color=colors,
)
axes[1].set(title="Total EFLOPs per test sample", ylabel="effective operations/sample")
axes[1].set_xticks(x, final_results["model"], rotation=15, ha="right")
axes[1].bar_label(
    total_bars,
    labels=[f"{value:,.0f}" for value in final_results["total EFLOPs/sample"]],
    padding=3,
)
axes[1].legend()
plt.show()

## 7. Direct comparison

The last cell reports differences rather than imposing an arbitrary threshold for “as good as.” With one run per model, these are descriptive comparisons, not confidence intervals or significance tests.

In [ ]:
lif_row = final_results.set_index("kind").loc["lif"]
comparison_rows = []
for kind in ("pif", "hetero_pif"):
    row = final_results.set_index("kind").loc[kind]
    comparison_rows.append(
        {
            "model": row["model"],
            "accuracy difference vs LIF": row["test accuracy"] - lif_row["test accuracy"],
            "total EFLOP ratio vs LIF": row["total EFLOPs/sample"] / lif_row["total EFLOPs/sample"],
            "total EFLOP reduction vs LIF (%)": 100 * (
                1 - row["total EFLOPs/sample"] / lif_row["total EFLOPs/sample"]
            ),
        }
    )

comparison = pd.DataFrame(comparison_rows)
display(comparison)
display(
    initialization_results[
        [
            "model",
            "empirical mean",
            "theoretical mean",
            "empirical std",
            "theoretical std",
            "relative std error (%)",
        ]
    ]
)